# Simulation Environment

## AMM Basics

## Tokens

We assume a set $T_0$ of **atomic token types**, representing native cryptocurrencies and application-specific tokens. Examples include:

- $\mathrm{ETH}$, the native cryptocurrency of Ethereum.
- $\mathrm{WBTC}$, Bitcoin wrapped as an ERC20 token.

A **minted token type** is an unordered pair of distinct atomic token types.  
If $\tau_0, \tau_1 \in T_0$ and $\tau_0 \neq \tau_1$, then the minted token type

$$
\{\tau_0, \tau_1\}
$$

represents shares in an AMM holding reserves of $\tau_0$ and $\tau_1$.

We denote by $T_1$ the set of minted token types, and define the universe of token types as:

$$
T = T_0 \cup T_1
$$

We use $\tau, \tau_0, \tau_1, \dots$ to range over $T$.

Tokens are **fungible**, meaning that units of the same type are interchangeable and divisible.

We write $r : \tau$ to denote $r \in \mathbb{R}_{\ge 0}$ units of token type $\tau$.

---

## Wallets

The wallet of a user $A$ is modeled as:

$$
A[\sigma]
$$

where $\sigma \in T \to \mathbb{R}_{\ge 0}$ is a finite partial map representing balances.

For $\tau \in T$, we write:

$$
\sigma(\tau)
$$

for the balance of token $\tau$ in wallet $A$.

---

## AMMs

An AMM holding reserves of tokens $\tau_0$ and $\tau_1$ (with $\tau_0 \neq \tau_1$) is modeled as:

$$
\{ r_0 : \tau_0, r_1 : \tau_1 \}
$$

Since order is immaterial:

$$
\{ r_0 : \tau_0, r_1 : \tau_1 \}
=
\{ r_1 : \tau_1, r_0 : \tau_0 \}
$$

---

## States

A state $\Gamma$ is a finite non-empty composition of wallets and AMMs:

$$
\Gamma =
A_1[\sigma_1]
\;|\;
\cdots
\;|\;
A_n[\sigma_n]
\;|\;
\{ r_1 : \tau_1, r'_1 : \tau'_1 \}
\;|\;
\cdots
$$

States are considered equivalent up to permutation of components.

---

## Token Supply

The total supply of token $\tau$ in state $\Gamma$, denoted $S_\Gamma^\tau$, is defined inductively:

### Wallet

$$
S_{A[\sigma]}^\tau =
\begin{cases}
\sigma(\tau) & \text{if } \tau \in \mathrm{dom}(\sigma) \\
0 & \text{otherwise}
\end{cases}
$$

### AMM

$$
S_{\{ r_0:\tau_0, r_1:\tau_1 \}}^\tau =
\begin{cases}
r_i & \text{if } \tau = \tau_i \\
0 & \text{otherwise}
\end{cases}
$$

### Composition

$$
S_{\Gamma_1 \mid \Gamma_2}^\tau
=
S_{\Gamma_1}^\tau
+
S_{\Gamma_2}^\tau
$$

---

## Transactions

State transitions are written:

$$
\Gamma \xrightarrow{T} \Gamma'
$$

where $T$ is one of the following.

---

### Deposit

$$
A : \mathrm{dep}(v_0 : \tau_0, v_1 : \tau_1)
$$

Transition:

$$
\frac{
\sigma(\tau_i) \ge v_i > 0
\quad
S_\Gamma^{\{\tau_0,\tau_1\}} = 0
}{
A[\sigma]
\mid
\Gamma
\xrightarrow{\mathrm{dep}}
A[\sigma - v_0:\tau_0 - v_1:\tau_1 + v:\{\tau_0,\tau_1\}]
\mid
\{ v_0:\tau_0, v_1:\tau_1 \}
\mid
\Gamma
}
$$

---

### Redeem

$$
A : \mathrm{rdm}(v : \{\tau_0,\tau_1\})
$$

Where:

$$
v_i = v \cdot \frac{r_i}{S_\Gamma^{\{\tau_0,\tau_1\}}}
\quad (i=0,1)
$$

Transition:

$$
\frac{
\sigma(\{\tau_0,\tau_1\}) \ge v > 0
\quad
v < S_\Gamma^{\{\tau_0,\tau_1\}}
}{
A[\sigma]
\mid
\{ r_0:\tau_0, r_1:\tau_1 \}
\mid
\Delta
\xrightarrow{\mathrm{rdm}}
A[\sigma + v_0:\tau_0 + v_1:\tau_1 - v:\{\tau_0,\tau_1\}]
\mid
\{ r_0 - v_0:\tau_0, r_1 - v_1:\tau_1 \}
\mid
\Delta
}
$$

---

### Swap

$$
A : \mathrm{swap}(x, \tau_0, \tau_1)
$$

Let:

$$
y = x \cdot S_X(x, r_0, r_1)
$$

Transition:

$$
\frac{
\sigma(\tau_0) \ge x
\quad
y = x \cdot S_X(x,r_0,r_1)
\quad
y < r_1
}{
A[\sigma]
\mid
\{ r_0:\tau_0, r_1:\tau_1 \}
\mid
\Gamma
\xrightarrow{\mathrm{swap}}
A[\sigma - x:\tau_0 + y:\tau_1]
\mid
\{ r_0 + x:\tau_0, r_1 - y:\tau_1 \}
\mid
\Gamma
}
$$

In [ ]:
from dataclasses import dataclass
from typing import Dict, FrozenSet, Union, Optional, List
from abc import ABC, abstractmethod


# Tokens
@dataclass(frozen=True)
class AtomicToken:
    name: str

    def __repr__(self):
        return self.name


@dataclass(frozen=True)
class MintedToken:
    pair: FrozenSet[AtomicToken]

    def __repr__(self):
        t0, t1 = sorted(self.pair, key=lambda t: t.name)
        return f"LP({t0}-{t1})"


Token = Union[AtomicToken, MintedToken]


# Wallet
class Wallet:
    def __init__(self, owner: str):
        self.owner = owner
        self.balances: Dict[Token, float] = {}

    def deposit(self, token: Token, amount: float):
        self.balances[token] = self.balances.get(token, 0.0) + amount

    def withdraw(self, token: Token, amount: float):
        if self.balances.get(token, 0.0) < amount:
            raise ValueError(f"{self.owner} insufficient {token}")
        self.balances[token] -= amount

    def balance(self, token: Token) -> float:
        return self.balances.get(token, 0.0)


# Market Maker Interface
class MarketMaker(ABC):

    @abstractmethod
    def lp_minted(self, reserves, amounts, total_lp):
        pass

    @abstractmethod
    def swap_out(self, reserves, token_in, amount_in):
        pass

    @abstractmethod
    def redeem(self, reserves, lp_amount, total_lp):
        pass

# AMM
class AMM:
    def __init__(self, token0, token1, market_maker: MarketMaker, reserve0=0.0, reserve1=0.0):
        self.tokens = frozenset({token0, token1})
        self.reserves = {token0: reserve0, token1: reserve1}
        self.mm = market_maker
    def deposit(self, amounts):
        for t, a in amounts.items(): self.reserves[t] += a
    def withdraw(self, amounts):
        for t, a in amounts.items(): self.reserves[t] -= a

# -------- Transaction --------
@dataclass
class Transaction:
    type: str
    wallet: Wallet
    token0: Optional[AtomicToken] = None
    token1: Optional[AtomicToken] = None
    amount0: float = 0.0

class State:
    def __init__(self, wallets: List[Wallet], amms: List[AMM]):
        self.wallets = {w.owner: w for w in wallets}
        self.amms = amms
    def find_amm(self, t0, t1):
        return next(a for a in self.amms if a.tokens == frozenset({t0, t1}))
    def swap(self, tx: Transaction):
        amm = self.find_amm(tx.token0, tx.token1)
        out = amm.mm.swap_out(amm.reserves, tx.token0, tx.amount0)
        tx.wallet.withdraw(tx.token0, tx.amount0)
        amm.deposit({tx.token0: tx.amount0})
        amm.withdraw({tx.token1: out})
        tx.wallet.deposit(tx.token1, out)

# AMM Models

## Constant Product Market Maker

The constant product swap rate function is:

$$
S_X(x, r_0, r_1) = \frac{r_1}{r_0 + x}
$$

The constant product swap rate ensures that, if an AMM

$$
\{ r_0 : \tau_0, r_1 : \tau_1 \}
$$

evolves into

$$
\{ r_0 + x : \tau_0, r_1 - y : \tau_1 \}
$$

upon a swap, then the product between the reserves is preserved:

$$
(r_0 + x)(r_1 - y)
=
(r_0 + x)
\left(
r_1 - x \cdot \frac{r_1}{r_0 + x}
\right)
=
r_0 r_1
$$

Here we distinguish between Uniswap V1 (ETH ↔ TOKEN pairs only) and Uniswap V2 (TOKEN ↔ TOKEN pairs), while both versions use the constant product invariant.

---

## Constant Sum Market Maker

The constant sum swap rate function is:

$$
S_X(x, r_0, r_1) = 1
$$

The constant sum swap rate ensures that, if an AMM

$$
\{ r_0 : \tau_0, r_1 : \tau_1 \}
$$

evolves into

$$
\{ r_0 + x : \tau_0, r_1 - y : \tau_1 \}
$$

upon a swap, then the sum of the reserves is preserved:

$$
(r_0 + x) + (r_1 - y)
=
(r_0 + x) + (r_1 - x)
=
r_0 + r_1
$$

---

## Constant Mean Market Maker

The constant mean swap rate function is defined as:

$$
S_X(x, r_0, r_1)
=
\frac{
(r_0 + x)^{w_0 - 1}
(r_1)^{w_1}
}{
(r_0 + x)^{w_0}
(r_1)^{w_1}
}
$$

where $w_0, w_1 > 0$ and $w_0 + w_1 = 1$.

The constant mean swap rate ensures that, if an AMM

$$
\{ r_0 : \tau_0, r_1 : \tau_1 \}
$$

evolves into

$$
\{ r_0 + x : \tau_0, r_1 - y : \tau_1 \}
$$

upon a swap, then the weighted geometric mean of the reserves is preserved:

$$
(r_0 + x)^{w_0}
(r_1 - y)^{w_1}
=
r_0^{w_0}
r_1^{w_1}
$$


In [ ]:
# Uniswap V1 CPMM
class UniswapV1(MarketMaker):
    """
    CPMM x*y=k
    Always ETH <-> TOKEN
    """

    def lp_minted(self, reserves, amounts, total_lp):
        tokens = list(reserves.keys())
        eth = [t for t in tokens if t.name == "ETH"][0]
        tok = [t for t in tokens if t != eth][0]

        if total_lp == 0:
            # Pool creation
            return (amounts[eth] * amounts[tok]) ** 0.5

        share = amounts[eth] / reserves[eth]
        return share * total_lp

    def swap_out(self, reserves, token_in, amount_in):
        tokens = list(reserves.keys())
        token_out = [t for t in tokens if t != token_in][0]

        x = reserves[token_in]
        y = reserves[token_out]

        k = x * y
        new_x = x + amount_in
        new_y = k / new_x

        return y - new_y

    def redeem(self, reserves, lp_amount, total_lp):
        share = lp_amount / total_lp
        return {t: reserves[t] * share for t in reserves}


# Uniswap V2 CPMM
class UniswapV2(MarketMaker):
    def __init__(self, fee=0.01):
        self.fee = fee

    def lp_minted(self, reserves, amounts, total_lp):
        tokens = list(reserves.keys())
        if total_lp == 0:
            return (amounts[tokens[0]] * amounts[tokens[1]]) ** 0.5
        share = min(amounts[t] / reserves[t] for t in tokens)
        return share * total_lp

    def swap_out(self, reserves, token_in, amount_in):
        token_out = [t for t in reserves.keys() if t != token_in][0]
        x, y = reserves[token_in], reserves[token_out]

        amount_in_with_fee = amount_in * (1 - self.fee)
        return y - (x * y) / (x + amount_in_with_fee)

    def redeem(self, reserves, lp_amount, total_lp):
        share = lp_amount / total_lp
        return {t: reserves[t] * share for t in reserves}

# CSMM
class CSMM(MarketMaker):
    """
    x + y = k
    Constant price, no slippage
    """

    def lp_minted(self, reserves, amounts, total_lp):
        tokens = list(reserves.keys())
        t0, t1 = tokens[0], tokens[1]

        if total_lp == 0:
            return amounts[t0] + amounts[t1]

        share = amounts[t0] / reserves[t0]
        return share * total_lp

    def swap_out(self, reserves, token_in, amount_in):
        tokens = list(reserves.keys())
        token_out = [t for t in tokens if t != token_in][0]

        price = reserves[token_out] / reserves[token_in]
        return amount_in * price

    def redeem(self, reserves, lp_amount, total_lp):
        share = lp_amount / total_lp
        return {t: reserves[t] * share for t in reserves}


# CMMM (Constant Mean) 
class CSMM(MarketMaker):
    def __init__(self, price: float):
        self.price = price  # P fijo del pool

    def lp_minted(self, reserves, amounts, total_lp):
        tokens = list(reserves.keys())
        t0, t1 = tokens[0], tokens[1]

        if total_lp == 0:
            return amounts[t0] + amounts[t1] / self.price

        share = amounts[t0] / reserves[t0]
        return share * total_lp

    def swap_out(self, reserves, token_in, amount_in):
        tokens = list(reserves.keys())
        token_out = [t for t in tokens if t != token_in][0]

        if token_in.name == "ETH":
            # ETH -> DAI
            return amount_in * self.price
        else:
            # DAI -> ETH
            return amount_in / self.price

    def redeem(self, reserves, lp_amount, total_lp):
        share = lp_amount / total_lp
        return {t: reserves[t] * share for t in reserves}


# HFMM (Hybrid Function Market Maker)
class HFMM(MarketMaker):
    def __init__(self, lmbda: float = 0.5, p_init: float = 2500.0):
        self.lmbda = lmbda
        self.p = p_init

    def _calculate_k(self, x, y):
        y_norm = y / self.p
        return self.lmbda * (x + y_norm) / 2 + (1 - self.lmbda) * (x * y_norm)**0.5

    def lp_minted(self, reserves, amounts, total_lp):
        tokens = list(reserves.keys())
        if total_lp == 0:
            return self._calculate_k(amounts[tokens[0]], amounts[tokens[1]])
        k_old = self._calculate_k(reserves[tokens[0]], reserves[tokens[1]])
        k_new = self._calculate_k(reserves[tokens[0]] + amounts[tokens[0]], 
                                  reserves[tokens[1]] + amounts[tokens[1]])
        return ((k_new - k_old) / k_old) * total_lp

    def swap_out(self, reserves, token_in, amount_in):
        tokens = list(reserves.keys())
        is_token_eth = (token_in.name == "ETH")
        x_old, y_old = reserves[tokens[0]], reserves[tokens[1]] # eth, dai
        
        k_target = self._calculate_k(x_old, y_old)
        
        if is_token_eth:
            x_new = x_old + amount_in
            y_new_val = self._solve_for_y(x_new, k_target)
            return max(0, y_old - y_new_val)
        else:
            y_new = y_old + amount_in
            x_new_val = self._solve_for_x(y_new, k_target)
            return max(0, x_old - x_new_val)

    def _solve_for_y(self, x, k):
        y_guess = k * self.p
        for _ in range(20):
            y_n = y_guess / self.p
            f = self.lmbda * (x + y_n) / 2 + (1 - self.lmbda) * (x * y_n)**0.5 - k
            df = self.lmbda / (2 * self.p) + (1 - self.lmbda) * 0.5 * (x / (y_guess * self.p))**0.5
            y_guess -= f / df
            if abs(f) < 1e-8: break
        return y_guess

    def _solve_for_x(self, y, k):
        x_guess = k
        y_n = y / self.p
        for _ in range(20):
            f = self.lmbda * (x_guess + y_n) / 2 + (1 - self.lmbda) * (x_guess * y_n)**0.5 - k
            df = self.lmbda / 2 + (1 - self.lmbda) * 0.5 * (y_n / x_guess)**0.5
            x_guess -= f / df
            if abs(f) < 1e-8: break
        return x_guess

    def redeem(self, reserves, lp_amount, total_lp):
        share = lp_amount / total_lp
        return {t: reserves[t] * share for t in reserves}


## Main actors in an AMM

Although there exists more actors in an AMM, we will only take into account the following: Liquidity Providers, traders and arbitrageurs.

### Liquidity Providers (LPs)

Liquidity Providers are users who deposit assets into an Automated Market Maker (AMM) pool, creating the liquidity necessary for trades to occur. In exchange for providing liquidity, LPs earn a share of the transaction fees generated by the pool, which compensates them for making the assets available. However, they are exposed to impermanent loss, which happens when the relative prices of the tokens in the pool change compared to when they were initially deposited. LPs play a fundamental role in the functioning of the AMM since without their deposits, the market would not be able to operate efficiently.

### Traders

Traders are participants who swap one token for another within the AMM pool. They provide the demand that drives the automated pricing mechanism and benefit from instant trades without needing a direct counterparty. Each swap involves paying a small transaction fee, which is distributed to the liquidity providers. By interacting with the pool, traders influence the token price ratios, and their activity can indirectly create opportunities for arbitrageurs when prices diverge from the broader market.

### Arbitrageurs

Arbitrageurs are specialized participants who exploit price differences between the AMM pool and external markets. They execute trades that bring the AMM price in line with the market price, earning profit from these discrepancies. Although arbitrageurs do not provide liquidity or hold positions for long-term investment, their activity ensures that the pool maintains price efficiency. By constantly correcting imbalances, arbitrageurs contribute to a stable and fair pricing mechanism within the AMM.


### Two-Point Arbitrage

**Concept:**  
Two-point arbitrage occurs when the **same asset has different prices across two markets** (e.g., an AMM vs. a centralized exchange). Traders can buy in the cheaper market and sell in the more expensive one to earn risk-free profits, until supply and demand forces align the prices.

---

#### Basic Mechanism

1. **External Market Price**: $M_{Y/X}$ = price of 1 unit of X (ETH) in terms of Y (ABC) on the external exchange.  
2. **AMM Price**: $P_{Y/X}$ = price of 1 unit of X in the AMM.  
3. **Arbitrage Opportunity**:  
   - If $P^a_{Y/X} > M^b_{Y/X}$: buy X externally, sell in AMM → profit per unit = $P^a_{Y/X} - M^b_{Y/X}$.  
   - If $P^b_{Y/X} < M^a_{Y/X}$: sell X externally, buy in AMM → profit exists in the opposite direction.

---

#### Effect of Arbitrage

- Arbitrageurs trade until the **no-arbitrage condition** is satisfied: prices converge, and opportunities disappear.  
- With **trading fees** ($\tau$) on the AMM, the bid-ask spread increases:
  - $p^b_{Y/X} = P_\text{bid}$ (AMM buys X)
  - $p^a_{Y/X} = P_\text{ask}$ (AMM sells X)  
- Arbitrage is only profitable if $M_{Y/X}$ lies **outside the fee-adjusted AMM range**.  

---

#### Key Observations

1. Discrete trades widen the bid-ask spread, reducing arbitrage opportunities.  
2. Higher trading fees or transaction costs also **limit arbitrage profits**.  
3. Incorporating **external bid/ask rates** requires adjusting the no-arbitrage bounds:  
   $$
   p^b_{Y/X} \le m^a_{Y/X}, \quad m^b_{Y/X} \le p^a_{Y/X}
   $$

**Conclusion:**  
Two-point arbitrage enforces **price alignment across markets** and is a natural mechanism that keeps AMM prices in check relative to external markets.


## Simulation of a CPMM (Uniswap v2)

In [ ]:
import numpy as np
import random

SEED = 1
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
import requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# 1. DATA ACQUISITION & SIMULATION SETUP

filename = "eth_market_data.csv"
if not os.path.exists(filename):
    url = "https://api.coingecko.com/api/v3/coins/ethereum/market_chart"
    response = requests.get(url, params={"vs_currency": "usd", "days": "1"}).json()
    df = pd.DataFrame(response["prices"], columns=["timestamp", "price_dai"])
    df.to_csv(filename, index=False)
else:
    df = pd.read_csv(filename)

df["timestamp"] = pd.to_datetime(df["timestamp"], unit='ms')

# Token and Wallet Setup
eth, dai = AtomicToken("ETH"), AtomicToken("DAI")
arb_wallet = Wallet("Arbitrageur")
trader_wallet = Wallet("NoiseTraders")

initial_eth, initial_dai = 1000.0, 1000000.0
arb_wallet.deposit(eth, initial_eth)
arb_wallet.deposit(dai, initial_dai)
trader_wallet.deposit(eth, 1000.0)
trader_wallet.deposit(dai, 2500000.0)

# Speed and Step settings
frame_duration = 200     
transition_duration = 200 
fee = 0.01  
step_skip = 2
p_init = df["price_dai"].iloc[0]
x_start = 10.0

amm_pool = AMM(eth, dai, UniswapV2(), reserve0=x_start, reserve1=x_start * p_init)
sim_state = State([arb_wallet, trader_wallet], [amm_pool])

# 2. SIMULATION EXECUTION WITH IL & FEES

history = []

# Inicial para cálculo de IL y fees
initial_eth_pool, initial_dai_pool = amm_pool.reserves[eth], amm_pool.reserves[dai]
total_fees = 0.0  # acumulador de fees

for i in range(len(df)):
    m_price = df["price_dai"].iloc[i]
    res = amm_pool.reserves

    # Noise Trades (Market Swings) con cálculo de fees
    if np.random.random() < 0.5:
        direction = np.random.choice(["buy", "sell"])
        if direction == "buy":
            amount_dai = res[dai] * np.random.uniform(0.0005, 0.005)
            sim_state.swap(Transaction("swap", trader_wallet, dai, eth, amount_dai))
            total_fees += amount_dai * fee  # sumar fees en DAI
        else:
            amount_eth = res[eth] * np.random.uniform(0.0005, 0.005)
            sim_state.swap(Transaction("swap", trader_wallet, eth, dai, amount_eth))
            total_fees += amount_eth * fee * m_price  # convertir fees a DAI

    # Arbitrage Logic
    res = amm_pool.reserves
    p_amm = res[dai] / res[eth]
    p_bid, p_ask = p_amm * (1 - fee), p_amm / (1 - fee)
    
    if m_price > p_ask:
        target_x = np.sqrt((res[eth]*res[dai]) / (m_price * (1-fee)))
        dy_in = (res[eth]*res[dai] / target_x) - res[dai]
        if dy_in > 0.001: 
            sim_state.swap(Transaction("swap", arb_wallet, dai, eth, dy_in))
    elif m_price < p_bid:
        target_x = np.sqrt((res[eth]*res[dai]) / (m_price / (1-fee)))
        dx_in = target_x - res[eth]
        if dx_in > 0.001: 
            sim_state.swap(Transaction("swap", arb_wallet, eth, dai, dx_in))

    res = amm_pool.reserves
    current_k = res[eth] * res[dai]
    m_x = np.sqrt(current_k / m_price)

    theta = (res[dai] / res[eth]) / p_init
    il_val = (2 * np.sqrt(theta) / (1 + theta) - 1)

    # Net Return
    v_pool = res[eth] * m_price + res[dai]
    v_hodl = initial_eth_pool * m_price + initial_dai_pool
    net_val = (v_pool / v_hodl) - 1

    history.append({
        'x': res[eth], 'y': res[dai], 'k': current_k,
        'm_x': m_x, 'm_y': current_k / m_x,
        'bid_x': res[eth] * np.sqrt(1/(1-fee)), 
        'ask_x': res[eth] * np.sqrt(1-fee),
        'IL': il_val,
        'fees': total_fees,
        'net_return': net_val
    })

h_df = pd.DataFrame(history)

# 3. DYNAMIC VISUALIZATION WITH IL, FEES & NET RETURN

y_min, y_max = df["price_dai"].min() * 0.97, df["price_dai"].max() * 1.03
x_time_min, x_time_max = df["timestamp"].min(), df["timestamp"].max()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("<b>External Market Price</b>", "<b>CPMM Curve: k = x*y (Uniswap V2)</b>"),
    horizontal_spacing=0.15
)

x_range = np.linspace(h_df['x'].min()*0.8, h_df['x'].max()*1.2, 200)

# Market Price
fig.add_trace(go.Scatter(
    x=[df["timestamp"][0]], y=[df["price_dai"][0]],
    mode='lines', line=dict(color='blue'),
    name="Market Price (DAI)"
), row=1, col=1)

# Invariant Curve
fig.add_trace(go.Scatter(
    x=x_range, y=h_df['k'][0]/x_range,
    line=dict(color='rgba(0,0,0,0.1)', width=1),
    name="Invariant Curve (k)"
), row=1, col=2)

# Ext. Market State
fig.add_trace(go.Scatter(
    x=[h_df['m_x'][0]], y=[h_df['m_y'][0]], mode='markers',
    marker=dict(color='blue', size=8),
    name="Ext. Market State"
), row=1, col=2)

# Fee Band
fig.add_trace(go.Scatter(
    x=[h_df['bid_x'][0]], y=[h_df['k'][0]/h_df['bid_x'][0]], mode='markers',
    marker=dict(color='orange', symbol='line-ns-open'),
    name="Fee Band"
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=[h_df['ask_x'][0]], y=[h_df['k'][0]/h_df['ask_x'][0]], mode='markers',
    marker=dict(color='orange', symbol='line-ns-open'),
    showlegend=False
), row=1, col=2)

# Pool State
fig.add_trace(go.Scatter(
    x=[h_df['x'][0]], y=[h_df['y'][0]], mode='markers',
    marker=dict(color='red', size=12, line=dict(color='white', width=1)),
    name="Pool State",
    legendgroup="pool"
), row=1, col=2)

# IL en leyenda (4 decimales)
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    showlegend=True,
    name=f"IL: {h_df['IL'][0]:.4%}",
    legendgroup="pool"
))

# Fees en leyenda
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    showlegend=True,
    name=f"Fees: {h_df['fees'][0]:.2f} DAI",
    legendgroup="pool"
))

# Net Return en leyenda
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    showlegend=True,
    name=f"Net Return: {h_df['net_return'][0]:.4%}",
    legendgroup="pool"
))

# ----------------------
# AXIS LABELS
# ----------------------
fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_yaxes(title_text="Price (DAI/ETH)", row=1, col=1)
fig.update_xaxes(title_text="ETH Reserves (x)", row=1, col=2)
fig.update_yaxes(title_text="DAI Reserves (y)", row=1, col=2)

# ----------------------
# FRAMES FOR ANIMATION
# ----------------------
frames = []
for i in range(0, len(df), step_skip):
    frames.append(go.Frame(data=[
        # Market Price
        go.Scatter(x=df["timestamp"][:i+1], y=df["price_dai"][:i+1]),
        # Invariant Curve
        go.Scatter(x=x_range, y=h_df['k'][i]/x_range),
        # Ext. Market State
        go.Scatter(x=[h_df['m_x'][i]], y=[h_df['m_y'][i]]),
        # Fee Band
        go.Scatter(x=[h_df['bid_x'][i]], y=[h_df['k'][i]/h_df['bid_x'][i]]),
        go.Scatter(x=[h_df['ask_x'][i]], y=[h_df['k'][i]/h_df['ask_x'][i]]),
        # Pool State
        go.Scatter(x=[h_df['x'][i]], y=[h_df['y'][i]]),
        # IL, Fees y Net Return en frames
        go.Scatter(x=[None], y=[None], name=f"IL: {h_df['IL'][i]:.4%}"),
        go.Scatter(x=[None], y=[None], name=f"Fees: {h_df['fees'][i]:.2f} DAI"),
        go.Scatter(x=[None], y=[None], name=f"Net Return: {h_df['net_return'][i]:.4%}")
    ], name=f"f{i}"))

fig.frames = frames

fig.update_layout(
    height=600, template="plotly_white",
    xaxis1=dict(range=[x_time_min, x_time_max], type="date", fixedrange=True),
    yaxis1=dict(range=[y_min, y_max], fixedrange=True),
    xaxis2=dict(range=[h_df['x'].min()*0.95, h_df['x'].max()*1.05], fixedrange=True),
    yaxis2=dict(range=[h_df['y'].min()*0.95, h_df['y'].max()*1.05], fixedrange=True),
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        direction="right",
        buttons=[dict(
            label="▶ Start Simulation",
            method="animate",
            args=[None, {
                "frame": {"duration": frame_duration, "redraw": True},
                "fromcurrent": True,
                "transition": {"duration": transition_duration, "easing": "quadratic-in-out"},
                "mode": "immediate"
            }]
        )]
    )]
)

fig.show()


![CPMM Simulation](cpmm_uniswap_simulation.gif)

We have simulated a Constant Product Market Maker (CPMM) considering an external market where the price of ETH in DAI is fluctuating. The CPMM curve illustrates, at each point, the state of the pool (showing where the pool currently is on the curve), the state of the external market (indicating where the pool would be if the reserves matched the external price), and the fee band, which defines the range within which arbitrage is not profitable.

In this simulation, a liquidity provider (LP) coexists with a trader who operates randomly and an arbitrageur. Some of the most interesting metrics displayed in the animation are the following:

**Impermanent Loss (IL):** the unrealized loss of the LP relative to the external market. Arbitrageurs push the LP to buy high and sell low, generating this temporary loss.

**Fees:** the amount of tokens (DAI, in this case) paid by each trade. These fees represent the LP's profit.

**Net Return:** the overall return obtained by the LP. Fees contribute positively to the net return, while impermanent loss caused by arbitrage reduces it.


## Simulation of a CSMM

In [ ]:
import requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# TOKENS AND WALLET
ETH = AtomicToken("ETH")
DAI = AtomicToken("DAI")
arb = Wallet("arbitrageur")
arb.deposit(ETH, 1e9)
arb.deposit(DAI, 1e9)

# DATA
filename = "eth_market_data.csv"
if not os.path.exists(filename):
    url = "https://api.coingecko.com/api/v3/coins/ethereum/market_chart"
    response = requests.get(url, params={"vs_currency":"usd","days":"1"}).json()
    df = pd.DataFrame(response["prices"], columns=["timestamp","price_dai"])
    df.to_csv(filename,index=False)
else:
    df = pd.read_csv(filename)
df["timestamp"] = pd.to_datetime(df["timestamp"], unit='ms')
p_fixed = df["price_dai"].iloc[0]  # Initial pool price
fee = 0.01
step_skip = 2
frame_duration = 50

# AMM AND STATE
mm = CSMM(price=p_fixed)
amm = AMM(ETH, DAI, mm, reserve0=500.0, reserve1=500.0*p_fixed)
state = State([arb],[amm])

def pool_price(amm, t0, t1):
    return amm.reserves[t1] / amm.reserves[t0]

# SIMULATION
history=[]
for i in range(len(df)):
    m_price = df["price_dai"].iloc[i]  # Market price
    p_pool = pool_price(amm, ETH, DAI)
    # Empty the AMM instantly if there is a price discrepancy
    if m_price > p_pool*(1+fee) and amm.reserves[DAI]>0:
        tx = Transaction(type="swap", wallet=arb, token0=DAI, token1=ETH, amount0=amm.reserves[DAI])
        state.swap(tx)
    elif m_price < p_pool*(1-fee) and amm.reserves[ETH]>0:
        tx = Transaction(type="swap", wallet=arb, token0=ETH, token1=DAI, amount0=amm.reserves[ETH])
        state.swap(tx)
    history.append({'x':amm.reserves[ETH],'y':amm.reserves[DAI],'m_p':m_price,'p_pool':p_pool})

h_df = pd.DataFrame(history)

# VISUALIZATION
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("<b>External Market Price</b>", "<b>CSMM Curve: k = x+y</b>"),
    horizontal_spacing=0.15
)

# Market price trace
fig.add_trace(go.Scatter(x=[df["timestamp"][0]], y=[df["price_dai"][0]], mode='lines', 
                         line=dict(color='blue'), name="Market Price (DAI)"), row=1, col=1)

# Initial pool state marker
fig.add_trace(
    go.Scatter(x=[h_df['x'][0]], y=[h_df['y'][0]], mode='markers',
               marker=dict(size=14, line=dict(width=2,color='white')), name="Pool State"),
    row=1, col=2
)

# Invariant line x*P + y = k
x_line = np.linspace(0, max(h_df['x'])*1.2,100)
y_line = (amm.reserves[ETH]*p_fixed + amm.reserves[DAI]) - x_line*p_fixed
fig.add_trace(
    go.Scatter(x=x_line, y=y_line, mode='lines', line=dict(color='rgba(0,0,0,0.1)', width=1), name="Invariant Line"),
    row=1, col=2
)

# Animation frames
frames=[go.Frame(data=[
    go.Scatter(x=df["timestamp"][:i+1], y=df["price_dai"][:i+1], mode='lines'),
    go.Scatter(x=[h_df['x'][i]], y=[h_df['y'][i]], mode='markers')
], name=f"f{i}") for i in range(0,len(h_df),step_skip)]
fig.frames = frames

# Axis limits
time_min, time_max = df["timestamp"].min(), df["timestamp"].max()
y_price_min, y_price_max = df["price_dai"].min()*0.98, df["price_dai"].max()*1.02
x_max = max(h_df['x'])*1.2
y_max = max(h_df['y'])*1.2

x_min_plot = 400  # Fixed x-axis minimum
x_max_plot = 1000 # Fixed x-axis maximum

fig.update_layout(
    height=600, template="plotly_white",
    xaxis1=dict(range=[time_min,time_max], type="date", title="Time"),
    yaxis1=dict(range=[y_price_min,y_price_max], title="Price (DAI/ETH)"),
    xaxis2=dict(range=[x_min_plot,x_max_plot], title="ETH Reserves (x)"),  # Fixed X-axis
    yaxis2=dict(range=[0,y_max], title="DAI Reserves (y)"),
    updatemenus=[dict(
        type="buttons", showactive=False, x=0, y=1.15,
        buttons=[dict(label="▶ Start Simulation", method="animate",
                      args=[None, {"frame":{"duration":frame_duration,"redraw":True}}])]
    )]
)

fig.show()



![CSMM Simulation](csmm_simulation.gif)

In the visualization, we observe that at a certain point the pool depletes one of its token reserves (DAI), rendering the pool non-functional.

### Theorical background

What we can observe in the animation is that there is a point where DAI instantly goes to 0. This happens because of the arbitrageurs, but why does it occur?

In a Constant Sum Market Maker (CSMM), the invariant is

$$
x + y = k,
$$

where \(x\) and \(y\) denote the reserves of tokens \(X\) and \(Y\), respectively.

Differentiating implicitly,

$$
\frac{dy}{dx} = -1.
$$

Therefore, the marginal price of \(Y\) in terms of \(X\) is

$$
P_{Y/X} = -\frac{dy}{dx} = 1.
$$

More generally, if the invariant is written as

$$
k = P x + y,
$$

then

$$
\frac{dy}{dx} = -P,
\qquad
P_{Y/X} = -\frac{dy}{dx} = P.
$$

Hence, the marginal price is constant and does not depend on the reserve levels. The pricing curve is linear.

### Arbitrage Mechanism

Let $m_{Y/X}$ denote the external market price, which evolves over time. The no-arbitrage condition requires:

$$
p^b_{Y/X} \le m^a_{Y/X},
\qquad
m^b_{Y/X} \le p^a_{Y/X}.
$$

If the external price deviates from the constant pool price \(P\), an arbitrage opportunity arises.

Suppose, for example, that

$$
P > m_{Y/X}.
$$

Then the pool overvalues token \(Y\) relative to the external market. An arbitrageur can:

1. Buy \(Y\) in the external market at price \(m_{Y/X}\),
2. Sell \(Y\) to the pool at price \(P\),
3. Extract \(X\) from the pool.

Since the CSMM price does not adjust with trade size, each infinitesimal trade yields strictly positive profit as long as $P \neq m_{Y/X}$.

### Reserve Depletion

Because the marginal price is constant, the arbitrage condition remains satisfied throughout the entire trading process. There is no endogenous price adjustment mechanism that restores equilibrium.

Formally, as long as

$$
P \ne m_{Y/X},
$$

the profit per unit traded is constant and non-zero. Therefore, the arbitrageur will optimally trade until one of the reserves reaches its boundary:

$$
x = 0
\quad \text{or} \quad
y = 0.
$$

In this specific scenario, when the price in the external market goes down, this is satisfied:

$$
p^b_{DAI/ETH} \ge m^a_{DAI/ETH},
$$

the pool overprices DAI relative to the external market. Arbitrageurs will continuously sell DAI to the pool and withdraw ETH. Since the price inside the CSMM remains fixed, this process continues until the pool is entirely depleted of one asset (DAI reaches zero).

### Conclusion

A CSMM is fundamentally unstable under price discrepancies because its linear invariant implies a constant marginal price. Unlike a Constant Product Market Maker (CPMM), there is no curvature to enforce price impact. Consequently, any deviation between the pool price and the external market price leads to complete reserve exhaustion through arbitrage.

For this reason, we will mainly focus on CPMMs, as their nonlinear pricing mechanism adjusts automatically and makes them better suited to volatile market conditions.

